# 🚦 Road Video Traffic Sign Recognition & Voice Alerts

This is the **standalone notebook version** of the video sign detection code, using your trained **`gtsrb_best_model.pth`** MobileNetV3-Large classifier. No separate `.py` file is needed.

**Run from top to bottom:**
1. Install/check libraries (if necessary).
2. Configure your video and model paths.
3. Load the trained classifier.
4. Preview sign candidates from the first frame.
5. Play the full video with visual / voice alerts and optionally save it.

**Important:** Your GTSRB network **classifies cropped signs**. OpenCV's color/shape filtering finds possible signs in the video. This is a demo, **not a reliable or safety-certified driving-warning system**.

## Step 1 — Dependencies

In the **same virtual environment and notebook kernel** you used for model training, install OpenCV if missing:

```python
%pip install opencv-python
```

You also need PyTorch, torchvision, NumPy, Pillow, and Matplotlib, which were already required by your training notebook. **Restart the notebook kernel after installing packages**.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import Counter, deque
import csv
import platform
import subprocess
import time

import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
import matplotlib.pyplot as plt

print('PyTorch:', torch.__version__)
print('OpenCV:', cv2.__version__)

## Step 2 — Choose your model, video, and alert settings

Place your `.pth` file and MP4 next to this notebook, or enter absolute file paths. Set `USE_WEBCAM = True` to use the camera instead of the MP4.

In [ ]:
MODEL_PATH = Path('gtsrb_best_model.pth')
VIDEO_PATH = Path('road.mp4')          # Replace with the name of your driving video
USE_WEBCAM = False                   # True = webcam, False = MP4/AVI
CAMERA_INDEX = 0

CONFIDENCE_THRESHOLD = 0.80          # Higher = fewer but more confident predictions
FRAME_STRIDE = 2                     # Classify candidate signs every 2nd frame
MIN_SIGN_SIZE = 18                   # Minimum proposal width and height, pixels
REQUIRED_CONFIRMATIONS = 3           # Repeated predictions before an alert
ALERT_COOLDOWN_SECONDS = 8.0

SHOW_VIDEO_WINDOW = True             # Shows an OpenCV player (works with local Jupyter)
ENABLE_VOICE_ALERTS = True           # macOS built-in `say` command
SAVE_ANNOTATED_VIDEO = True
OUTPUT_VIDEO_PATH = Path('annotated_road.mp4')
SAVE_ALERT_LOG = True
ALERT_LOG_PATH = Path('traffic_sign_alerts.csv')

print('Model:', MODEL_PATH.resolve(), 'Exists:', MODEL_PATH.is_file())
if not USE_WEBCAM:
    print('Video:', VIDEO_PATH.resolve(), 'Exists:', VIDEO_PATH.is_file())

## Step 3 — Load your existing trained classifier

The model architecture and class labels are read from your checkpoint; **no retraining** is performed.

In [ ]:
def device_for_machine():
    if torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

def load_classifier(path: Path, device):
    checkpoint = torch.load(str(path), map_location='cpu', weights_only=True)
    required = {'model_name', 'model_state_dict', 'class_names', 'num_classes',
                'image_size', 'mean', 'std'}
    missing = required.difference(checkpoint)
    if missing:
        raise ValueError(f'Checkpoint is missing fields: {sorted(missing)}')
    n = int(checkpoint['num_classes'])
    name = checkpoint['model_name']
    if name == 'mobilenet_v3_large':
        net = models.mobilenet_v3_large(weights=None)
        net.classifier[-1] = torch.nn.Linear(net.classifier[-1].in_features, n)
    elif name == 'resnet18':
        net = models.resnet18(weights=None)
        net.fc = torch.nn.Sequential(
            torch.nn.Linear(net.fc.in_features, 256),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.30),
            torch.nn.Linear(256, n),
        )
    else:
        raise ValueError(f'Unsupported architecture: {name!r}')
    net.load_state_dict(checkpoint['model_state_dict'], strict=True)
    net.to(device).eval()
    image_transform = transforms.Compose([
        transforms.Resize((int(checkpoint['image_size']),) * 2),
        transforms.ToTensor(),
        transforms.Normalize(checkpoint['mean'], checkpoint['std']),
    ])
    labels = list(checkpoint['class_names'])
    if len(labels) != n:
        raise ValueError('Number of class names does not match num_classes.')
    return net, image_transform, labels, checkpoint

DEVICE = device_for_machine()
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Move gtsrb_best_model.pth beside the notebook or edit MODEL_PATH: {MODEL_PATH.resolve()}')

model, image_transform, class_names, checkpoint = load_classifier(MODEL_PATH, DEVICE)
print('Device:', DEVICE)
print('Architecture:', checkpoint['model_name'])
print('Class count:', len(class_names))
print('Stored best validation accuracy:', checkpoint.get('best_val_accuracy', 'not present'))

## Step 4 — Detect sign-shaped candidates and classify them

The detector searches for plausible **red / blue** signs. It may miss signs with other colors or unusual lighting.

In [ ]:
def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix0, iy0 = max(ax, bx), max(ay, by)
    ix1, iy1 = min(ax+aw, bx+bw), min(ay+ah, by+bh)
    inter = max(0, ix1-ix0) * max(0, iy1-iy0)
    return inter / max(1, (aw*ah + bw*bh - inter))

def boxes_from_colors(bgr, min_size=16, max_regions=24):
    """Detect red/blue sign-shaped regions; returns (x,y,w,h) proposals.

    These are NOT learned detections and may mistake similarly colored objects.
    """
    height, width = bgr.shape[:2]
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    red_a = cv2.inRange(hsv, (0, 75, 65), (12, 255, 255))
    red_b = cv2.inRange(hsv, (168, 75, 65), (179, 255, 255))
    blue = cv2.inRange(hsv, (91, 75, 60), (135, 255, 255))
    masks = [cv2.bitwise_or(red_a, red_b), blue]

    proposals = []
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    for mask in masks:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            if min(w, h) < min_size or w > width * .42 or h > height * .60:
                continue
            aspect = w / max(h, 1)
            if not 0.65 <= aspect <= 1.48:
                continue
            area = cv2.contourArea(contour)
            fill = area / max(1, w * h)
            if not 0.12 <= fill <= 0.93:
                continue
            pad = int(0.12 * max(w, h))
            x0, y0 = max(0, x-pad), max(0, y-pad)
            x1, y1 = min(width, x+w+pad), min(height, y+h+pad)
            proposals.append((x0, y0, x1-x0, y1-y0))

    # Suppress near-duplicate candidates caused by separate red/blue contours.
    proposals.sort(key=lambda box: box[2] * box[3], reverse=True)
    selected = []
    for b in proposals:
        if any(iou(b, p) > .35 for p in selected):
            continue
        selected.append(b)
        if len(selected) >= max_regions:
            break
    return selected

def predict_crops(frame, boxes, model, image_transform, device):
    if not boxes:
        return []
    batch = []
    for x, y, w, h in boxes:
        rgb = cv2.cvtColor(frame[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
        batch.append(image_transform(Image.fromarray(rgb)))
    with torch.inference_mode():
        logits = model(torch.stack(batch).to(device))
        probabilities = torch.softmax(logits, dim=1)
        confidence, labels = probabilities.max(dim=1)
    return [(int(c), float(p)) for c, p in zip(labels.cpu(), confidence.cpu())]


## Step 5 — Test detections on one frame (recommended)

This quick preview helps check the model, video path, and OpenCV proposals before processing the entire video.

In [ ]:
source = CAMERA_INDEX if USE_WEBCAM else str(VIDEO_PATH)
cap_preview = cv2.VideoCapture(source)
if not cap_preview.isOpened():
    print(f'Cannot open video source: {source}. Fix VIDEO_PATH in Step 2.')
else:
    ok, first_frame = cap_preview.read()
    cap_preview.release()
    if ok:
        proposals = boxes_from_colors(first_frame, min_size=MIN_SIGN_SIZE)
        predictions = predict_crops(first_frame, proposals, model, image_transform, DEVICE)
        preview = first_frame.copy()
        for (x, y, w, h), (class_id, confidence) in zip(proposals, predictions):
            if confidence >= CONFIDENCE_THRESHOLD:
                cv2.rectangle(preview, (x, y), (x + w, y + h), (0, 255, 0), 2)
                cv2.putText(preview, f'{class_names[class_id]} {confidence:.0%}',
                            (x, max(20, y - 7)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, (0, 255, 0), 2)
        plt.figure(figsize=(13, 7))
        plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f'First frame — {len(proposals)} sign candidates')
        plt.show()
    else:
        print('Video opened but no frame could be read.')

## Step 6 — Temporal confirmation and voice alerts

A label must appear consistently over several analyzed frames before an alert is issued. macOS speech uses the built-in `say` utility.

In [ ]:
class StableAlerts:
    def __init__(self, required_hits=3, cooldown_seconds=8.0):
        self.required_hits = required_hits
        self.cooldown = cooldown_seconds
        self.tracks = []
        self.last_alerted = {}

    def update(self, detections, seconds):
        # Each detection is (box, class_id, confidence).
        for track in self.tracks:
            track['missed'] += 1
        for box, class_id, confidence in detections:
            best = None
            best_overlap = 0.25
            for track in self.tracks:
                overlap = iou(box, track['box'])
                if overlap > best_overlap and track['missed'] <= 5:
                    best = track
                    best_overlap = overlap
            if best is None:
                best = {'box': box, 'labels': deque(maxlen=8), 'missed': 0,
                        'last_fired': -1e9}
                self.tracks.append(best)
            best.update(box=box, missed=0)
            best['labels'].append(class_id)

        fired = []
        for track in self.tracks:
            if track['missed']:
                continue
            class_id, hits = Counter(track['labels']).most_common(1)[0]
            if hits < self.required_hits:
                continue
            if seconds - track['last_fired'] < self.cooldown:
                continue
            # Avoid repeated announcements if the same sign is visible for several seconds.
            if seconds - self.last_alerted.get(class_id, -1e9) < self.cooldown:
                continue
            track['last_fired'] = seconds
            self.last_alerted[class_id] = seconds
            fired.append(class_id)

        self.tracks = [t for t in self.tracks if t['missed'] <= 5]
        return fired

def speak_on_mac(message, current_process):
    if platform.system() != 'Darwin':
        return current_process
    if current_process is not None and current_process.poll() is None:
        return current_process
    return subprocess.Popen(['say', message], stdout=subprocess.DEVNULL,
                            stderr=subprocess.DEVNULL)

def draw_label(frame, text, x, y, color=(40, 220, 60)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = .55
    thickness = 2
    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    x = max(0, min(x, frame.shape[1]-tw-8))
    y = max(th + 6, y)
    cv2.rectangle(frame, (x, y-th-6), (x+tw+7, y+baseline+4), (10, 10, 10), -1)
    cv2.putText(frame, text, (x+3, y), font, scale, color, thickness,
                cv2.LINE_AA)


## Step 7 — Video processing pipeline

- Draw bounding boxes and traffic sign names
- Show an alert banner when a sign is confirmed
- Speak the sign name (optional, on macOS)
- Save annotated MP4 and CSV event log (optional)

In the video window, **Q** quits and **Space** pauses or resumes. If using a hosted notebook without a desktop GUI, set `SHOW_VIDEO_WINDOW = False` and review the saved file locally.

In [ ]:
def run(args):
    device = device_for_machine()
    if not args.model.is_file():
        raise FileNotFoundError(f'Cannot find trained model: {args.model.resolve()}')
    model, transform, class_names, meta = load_classifier(args.model, device)
    print(f"Loaded {meta['model_name']} ({len(class_names)} classes) on {device}")
    print(f"Checkpoint validation accuracy: {meta.get('best_val_accuracy', 'not stored')}%")
    print('Candidate detection uses OpenCV color/shape heuristics, not YOLO.')

    source = args.camera if args.video is None else str(args.video)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video source: {source}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not 1 <= fps <= 240:
        fps = 25.0
    size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    if 0 in size:
        size = None
    writer = None
    csv_handle = None
    csv_writer = None
    if args.events:
        csv_handle = args.events.open('w', newline='', encoding='utf-8')
        csv_writer = csv.writer(csv_handle)
        csv_writer.writerow(['video_time_seconds', 'sign_class', 'sign_name'])

    alerts = StableAlerts(required_hits=args.confirmations,
                          cooldown_seconds=args.cooldown)
    frame_no = 0
    latest_detections = []
    banner_text = ''
    banner_expire = 0.0
    tts_process = None
    paused = False
    try:
        while True:
            begin = time.monotonic()
            if paused:
                if not args.no_window:
                    key = cv2.waitKey(30) & 0xFF
                    if key == ord('q'): break
                    if key == ord(' '): paused = False
                continue
            ok, frame = cap.read()
            if not ok:
                break
            if size is None:
                size = (frame.shape[1], frame.shape[0])
            if args.save and writer is None:
                writer = cv2.VideoWriter(str(args.save),
                                         cv2.VideoWriter_fourcc(*'mp4v'), fps, size)
                if not writer.isOpened():
                    raise RuntimeError(f'Could not write video to: {args.save}')
            video_seconds = frame_no / fps
            if frame_no % args.stride == 0:
                boxes = boxes_from_colors(frame, min_size=args.min_size)
                results = predict_crops(frame, boxes, model, transform, device)
                latest_detections = [
                    (box, cls, conf) for box, (cls, conf) in zip(boxes, results)
                    if conf >= args.confidence
                ]
                for cls in alerts.update(latest_detections, video_seconds):
                    name = class_names[cls]
                    banner_text = f'ALERT: {name}'
                    banner_expire = video_seconds + 3.0
                    print(f'[{video_seconds:8.1f}s] {name}')
                    if csv_writer:
                        csv_writer.writerow([round(video_seconds, 2), cls, name])
                        csv_handle.flush()
                    if not args.no_speech:
                        tts_process = speak_on_mac(name, tts_process)

            for (x,y,w,h), cls, conf in latest_detections:
                cv2.rectangle(frame, (x,y), (x+w,y+h), (40,220,60), 2)
                draw_label(frame, f'{class_names[cls]} {conf:.0%}', x, y-5)
            if video_seconds < banner_expire:
                draw_label(frame, banner_text, 12, 30, (40,220,255))
            draw_label(frame, f'Video: {video_seconds:.1f}s  |  Q quit  SPACE pause',
                       12, frame.shape[0]-13, (220,220,220))

            if writer:
                writer.write(frame)
            if not args.no_window:
                cv2.imshow('Traffic Sign Recognition - Demo', frame)
                elapsed_ms = int((time.monotonic() - begin) * 1000)
                # For file videos, try to keep a reasonable playback rate.
                delay = max(1, int(1000/fps) - elapsed_ms) if args.video else 1
                key = cv2.waitKey(delay) & 0xFF
                if key == ord('q'):
                    break
                if key == ord(' '):
                    paused = True
            frame_no += 1
        print(f'Processed {frame_no} video frames.')
    finally:
        cap.release()
        if writer:
            writer.release()
        if csv_handle:
            csv_handle.close()
        if not args.no_window:
            cv2.destroyAllWindows()


## Step 8 — Run on the MP4 (or webcam)

This cell starts video processing. Make sure `VIDEO_PATH` exists if you're not using a webcam.

In [ ]:
settings = SimpleNamespace(
    video=None if USE_WEBCAM else VIDEO_PATH,
    camera=CAMERA_INDEX,
    model=MODEL_PATH,
    confidence=CONFIDENCE_THRESHOLD,
    stride=FRAME_STRIDE,
    min_size=MIN_SIGN_SIZE,
    confirmations=REQUIRED_CONFIRMATIONS,
    cooldown=ALERT_COOLDOWN_SECONDS,
    no_speech=not ENABLE_VOICE_ALERTS,
    no_window=not SHOW_VIDEO_WINDOW,
    save=OUTPUT_VIDEO_PATH if SAVE_ANNOTATED_VIDEO else None,
    events=ALERT_LOG_PATH if SAVE_ALERT_LOG else None,
)

if USE_WEBCAM or VIDEO_PATH.is_file():
    run(settings)
    if SAVE_ANNOTATED_VIDEO:
        print('Annotated MP4:', OUTPUT_VIDEO_PATH.resolve())
    if SAVE_ALERT_LOG:
        print('Alert history:', ALERT_LOG_PATH.resolve())
else:
    print(f'Add your MP4 beside the notebook or change VIDEO_PATH: {VIDEO_PATH.resolve()}')

## Step 9 — Review saved alerts (optional)

Displays the list of confirmed signs with the video timestamps. The output MP4 uses OpenCV's `mp4v` codec; some browser players may require conversion to H.264 to play inline.

In [ ]:
if ALERT_LOG_PATH.is_file():
    with ALERT_LOG_PATH.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print('Alerts recorded:', len(rows))
    for row in rows[:25]:
        print(f"{row['video_time_seconds']}s  —  {row['sign_name']}")
else:
    print('No event log yet. Run Step 8 with SAVE_ALERT_LOG = True.')